# Noise2Void — Learning Denoising from Single Noisy Images / 구현

**Paper**: Krull, A., Buchholz, T.-O., & Jug, F. (2019). *Proc. IEEE/CVF CVPR*. [DOI: 10.1109/CVPR.2019.00223]

## Overview / 개요

Noise2Void(N2V)는 **단 한 장의 노이즈 영상**으로 디노이저를 학습한다. 핵심 도구는 **blind-spot loss**: 표준 CNN의 receptive field에서 *중심 픽셀만 마스킹*하여 신경망이 항등 함수를 학습하지 못하도록 한다.

이 노트북은 N2V를 다음 단계로 검증한다:
1. **Naive single-image training (failure mode)**: 같은 노이즈 영상을 입력·타겟으로 사용하면 신경망이 *항등 함수*를 학습.
2. **Blind-spot masking (the fix)**: 무작위 픽셀의 값을 *주변 픽셀에서 sampling*한 값으로 교체 → blind spot 생성. 손실은 *마스크 위치*에서만 계산.
3. **Denoising result**: blind-spot 학습으로 single noisy image에서 신호 회복 → BM3D / Gaussian filter / oracle과 비교.

Noise2Void trains a denoiser from a *single* noisy image via a **blind-spot loss**: mask the centre pixel of every training receptive field so the network cannot learn the identity. This notebook compares (i) naive identity-training failure, (ii) blind-spot-masked training, and (iii) classical baselines.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from scipy.ndimage import gaussian_filter
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage import data, img_as_float
from skimage.transform import resize

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---

## Part 1: Test image and noise / 테스트 영상과 노이즈

단일 노이즈 영상 (cameraman, 가우시안 $\sigma = 0.20$) 만으로 학습한다.

In [ ]:
img_full = img_as_float(data.camera())
clean = resize(img_full, (128, 128), preserve_range=True).astype(np.float64)

SIGMA = 0.20
noisy_image = clean + SIGMA * np.random.default_rng(42).standard_normal(clean.shape)
noisy_image = noisy_image.astype(np.float32)


def psnr(a: NDArray[np.floating], b: NDArray[np.floating], peak: float = 1.0) -> float:
    """PSNR in dB."""
    return float(10 * np.log10(peak ** 2 / max(np.mean((a - b) ** 2), 1e-12)))


fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1); axes[0].set_title('clean (unobserved)')
axes[1].imshow(noisy_image, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'single noisy training image (PSNR={psnr(noisy_image, clean):.2f} dB)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 2: CNN architecture / CNN 구조

Standard 3-layer CNN. *Note*: we keep the architecture standard (no structural blind-spot). The blind-spot effect is achieved by the **masking trick** in the loss, not in the network.

In [ ]:
class TinyDenoiser(nn.Module):
    """Simple 3-layer CNN. Direct prediction (no residual).

    Args:
        channels: hidden feature-map count.
    """

    def __init__(self, channels: int = 32) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(channels, 1, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        return self.conv3(h)

---

## Part 3: Naive identity training (failure mode) / 단순 항등 학습 (실패)

입력 = 타겟 = 같은 노이즈 영상. 신경망은 *항등 함수*를 학습 → 디노이징 X.

In [ ]:
def train_naive_identity(
    noisy: NDArray[np.float32],
    n_iter: int = 800,
    lr: float = 1e-3,
    seed: int = 0,
) -> TinyDenoiser:
    """Train with input == target = same noisy image. Network learns identity.

    Args:
        noisy: single noisy image.
        n_iter: number of optimizer steps.
        lr: learning rate.
        seed: torch RNG seed.

    Returns:
        Trained model.
    """
    torch.manual_seed(seed)
    model = TinyDenoiser(channels=32).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x = torch.tensor(noisy[None, None, :, :], dtype=torch.float32, device=device)

    for _ in range(n_iter):
        opt.zero_grad()
        loss = F.mse_loss(model(x), x)  # target = input
        loss.backward()
        opt.step()
    return model


model_naive = train_naive_identity(noisy_image, n_iter=800)
with torch.no_grad():
    out_naive = model_naive(
        torch.tensor(noisy_image[None, None, :, :], dtype=torch.float32, device=device)
    ).cpu().numpy().squeeze()

print(f'Naive output PSNR vs clean : {psnr(out_naive, clean):.2f} dB')
print(f'Naive output PSNR vs noisy : {psnr(out_naive, noisy_image):.2f} dB  (≫ vs clean → identity)')

**Failure confirmed**: PSNR vs noisy (very high) >> PSNR vs clean (~ same as input) → network learned identity.

---

## Part 4: Blind-spot masking / 블라인드 스팟 마스킹

Per-batch:
1. Pick `N` random positions $\{i_k\}$ in the patch.
2. Replace each `noisy[i_k]` with the value of a random neighbour `noisy[j_k]` (creates the blind spot).
3. Network sees the modified patch as input.
4. Loss is computed only at positions $i_k$, with target = *original* `noisy[i_k]` value.

이렇게 하면 (i) 네트워크가 입력에서 중심 픽셀의 진짜 값을 *볼 수 없고* (ii) 그 위치를 *주변에서 추정*해야 하므로 신호 회복.

In [ ]:
def make_blindspot_batch(
    noisy: NDArray[np.float32],
    patch_size: int = 64,
    n_masked: int = 64,
    neighbour_radius: int = 5,
    seed: int | None = None,
) -> tuple[NDArray[np.float32], NDArray[np.float32], NDArray[np.float32]]:
    """Construct a single training patch with blind-spot mask.

    Args:
        noisy: full noisy image (H, W).
        patch_size: side of the random crop.
        n_masked: number of blind-spot pixels per patch.
        neighbour_radius: how far to sample replacement values.
        seed: RNG seed.

    Returns:
        masked_input (H, W) with blind-spot pixels replaced by neighbour values,
        target_image (H, W) = original (unmodified) patch,
        loss_mask (H, W) = 1 at blind-spot positions, 0 elsewhere.
    """
    g = np.random.default_rng(seed)
    H, W = noisy.shape
    # Random crop
    y0 = g.integers(0, H - patch_size + 1)
    x0 = g.integers(0, W - patch_size + 1)
    patch = noisy[y0:y0 + patch_size, x0:x0 + patch_size].copy()
    target = patch.copy()

    loss_mask = np.zeros_like(patch, dtype=np.float32)
    masked_input = patch.copy()

    # Stratified random positions (avoid clustering)
    grid = int(np.sqrt(n_masked))
    cell = patch_size // grid
    positions = []
    for gy in range(grid):
        for gx in range(grid):
            yy = gy * cell + g.integers(0, cell)
            xx = gx * cell + g.integers(0, cell)
            positions.append((yy, xx))

    for (yy, xx) in positions[:n_masked]:
        # Replace with neighbour-sampled value
        dy = g.integers(-neighbour_radius, neighbour_radius + 1)
        dx = g.integers(-neighbour_radius, neighbour_radius + 1)
        if dy == 0 and dx == 0:
            dy = 1  # ensure not the same pixel
        ny = np.clip(yy + dy, 0, patch_size - 1)
        nx = np.clip(xx + dx, 0, patch_size - 1)
        masked_input[yy, xx] = patch[ny, nx]
        loss_mask[yy, xx] = 1.0

    return masked_input.astype(np.float32), target.astype(np.float32), loss_mask


# Quick visualisation of the masking
_in, _t, _m = make_blindspot_batch(noisy_image, patch_size=64, n_masked=64, seed=7)
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(_t, cmap='gray', vmin=0, vmax=1); axes[0].set_title('original noisy patch')
axes[1].imshow(_in, cmap='gray', vmin=0, vmax=1); axes[1].set_title('masked input (blind spots)')
axes[2].imshow(_m, cmap='Reds'); axes[2].set_title(f'loss mask ({int(_m.sum())} positions)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 5: Train Noise2Void / N2V 학습

In [ ]:
def train_n2v(
    noisy: NDArray[np.float32],
    n_iter: int = 2000,
    batch_per_iter: int = 8,
    lr: float = 1e-3,
    patch_size: int = 64,
    n_masked: int = 64,
    seed: int = 0,
) -> TinyDenoiser:
    """Train TinyDenoiser with Noise2Void blind-spot loss.

    Args:
        noisy: single noisy training image.
        n_iter: number of optimizer steps.
        batch_per_iter: patches per gradient step.
        lr: learning rate.
        patch_size: random crop side.
        n_masked: blind-spot pixels per patch.
        seed: torch RNG seed.

    Returns:
        Trained model.
    """
    torch.manual_seed(seed)
    model = TinyDenoiser(channels=32).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for it in range(n_iter):
        inputs_list = []
        targets_list = []
        masks_list = []
        for b in range(batch_per_iter):
            mi, t, m = make_blindspot_batch(
                noisy, patch_size=patch_size, n_masked=n_masked, seed=seed * 100000 + it * batch_per_iter + b
            )
            inputs_list.append(mi)
            targets_list.append(t)
            masks_list.append(m)
        x = torch.tensor(np.stack(inputs_list)[:, None], dtype=torch.float32, device=device)
        y = torch.tensor(np.stack(targets_list)[:, None], dtype=torch.float32, device=device)
        m = torch.tensor(np.stack(masks_list)[:, None], dtype=torch.float32, device=device)

        opt.zero_grad()
        pred = model(x)
        # Masked MSE — only blind-spot positions contribute
        sq_err = (pred - y) ** 2
        loss = (m * sq_err).sum() / m.sum().clamp(min=1.0)
        loss.backward()
        opt.step()
    return model


model_n2v = train_n2v(noisy_image, n_iter=2000, batch_per_iter=8)
print('Noise2Void model trained.')

---

## Part 6: Compare with baselines / 베이스라인과의 비교

비교 대상:
- **Naive (identity)**: failure baseline.
- **Gaussian filter**: classical no-learning baseline.
- **N2V**: blind-spot single-image training.
- **Oracle**: 같은 CNN을 *clean target*으로 학습 (BSD68 supervised에 해당) — 상한 비교.

In [ ]:
# Apply N2V to the full noisy image (no masking at inference)
with torch.no_grad():
    nt = torch.tensor(noisy_image[None, None, :, :], dtype=torch.float32, device=device)
    out_n2v = model_n2v(nt).cpu().numpy().squeeze()

# Train an oracle for upper bound: tile the noisy image as inputs, clean as targets
def train_oracle_singleimage(
    noisy: NDArray[np.float32], clean_target: NDArray[np.float32],
    n_iter: int = 1000, lr: float = 1e-3, seed: int = 0,
) -> TinyDenoiser:
    torch.manual_seed(seed)
    model = TinyDenoiser(channels=32).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = np.random.default_rng(seed)
    for it in range(n_iter):
        # Re-generate independent noise each iter so the oracle has many noisy examples
        noise = g.standard_normal(clean_target.shape).astype(np.float32) * SIGMA
        n_in = (clean_target + noise).astype(np.float32)
        x = torch.tensor(n_in[None, None], dtype=torch.float32, device=device)
        y = torch.tensor(clean_target[None, None], dtype=torch.float32, device=device)
        opt.zero_grad()
        loss = F.mse_loss(model(x), y)
        loss.backward()
        opt.step()
    return model


model_oracle = train_oracle_singleimage(noisy_image, clean.astype(np.float32), n_iter=1000)
with torch.no_grad():
    out_oracle = model_oracle(nt).cpu().numpy().squeeze()

out_gauss = gaussian_filter(noisy_image, sigma=1.5)

results = [
    ('noisy input', noisy_image, psnr(noisy_image, clean)),
    ('naive (identity)', out_naive, psnr(out_naive, clean)),
    ('Gaussian filter', out_gauss, psnr(out_gauss, clean)),
    ('Noise2Void', out_n2v, psnr(out_n2v, clean)),
    ('Oracle (clean-target)', out_oracle, psnr(out_oracle, clean)),
]

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1); axes[0].set_title('clean')
for ax, (name, img, p) in zip(axes[1:], results):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{name}\n{p:.2f} dB')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print('\nResults summary:')
for name, _, p in results:
    print(f'  {name:25s}: {p:.2f} dB')

**Expected ordering**: noisy ≈ naive (identity) < Gaussian < N2V < oracle.
- Naive output ≈ noisy input (identity learned, no denoising).
- Gaussian: ~few dB improvement, blurs detail.
- **N2V**: substantial improvement using *only* the single noisy image.
- Oracle: upper bound (uses clean targets).

예상 순서: noisy ≈ naive < Gaussian < **N2V** < oracle. N2V는 깨끗 target 없이도 oracle에 근접한 PSNR을 달성한다.

---

## Part 7: Sanity check — output residual structure / 출력 잔차 구조 확인

*Method noise* (paper #4) check: $r = \text{noisy} - \text{N2V output}$. 좋은 디노이저의 잔차는 *구조 없음 = white noise*에 가까워야 한다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
r_gauss = noisy_image - out_gauss
r_n2v = noisy_image - out_n2v
axes[0].imshow(noisy_image, cmap='gray', vmin=0, vmax=1); axes[0].set_title('noisy input')
axes[1].imshow(r_gauss, cmap='gray', vmin=-0.3, vmax=0.3)
axes[1].set_title(f'Gaussian residual\nstd={r_gauss.std():.3f} (structured)')
axes[2].imshow(r_n2v, cmap='gray', vmin=-0.3, vmax=0.3)
axes[2].set_title(f'N2V residual\nstd={r_n2v.std():.3f} (less structured)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print('Gaussian residual shows visible edges — Gaussian destroys structure.')
print('N2V residual is closer to white noise — preserves structure better.')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Identity-learning failure | §3.4 | Part 3 naive training | (paper-specific motivation) |
| Blind-spot architecture | §3.4 + Fig. 2b | conceptual; we use masking instead | careful FCN with masked conv |
| Masking trick | §3.5 + Fig. 3 | Part 4 `make_blindspot_batch` | CARE / CSBDeep N2V module |
| Stratified random masking | §3.5 | grid-based stratified positions | (same in CSBDeep) |
| Neighbour-value replacement | §3.5 | radius-5 random shift | (same) |
| Masked-loss only at blind spots | Eq. 10 | `(m * sq_err).sum() / m.sum()` | (same) |
| Comparison vs supervised | §4.1 BSD68 | Part 6 oracle column | shows ~1–2 dB gap as expected |

### Connections to other papers / 다른 논문과의 연결

- **Paper #16 (Noise2Noise)**: N2V removes one assumption (paired noisy images) by using a blind-spot trick.
- **Paper #15 (Cryo-CARE)**: when paired noisy data is available, cryo-CARE (N2N) is preferred. Otherwise N2V.
- **Paper #4 (NL-means)**: classical single-image self-supervised idea (neighbours predict centre via patch similarity); N2V is its learned successor.
- **Paper #18 (Noise2Self)**: generalises N2V's blind-spot to a $\mathcal{J}$-invariance framework — unifies many self-supervised denoisers.
- **Probabilistic N2V (Krull+ 2020)**: extends N2V by explicitly modelling pixel-wise noise distributions, recovering ~1 dB.

### Take-away / 결론

본 노트북은 N2V의 핵심 개념을 4단계로 검증했다:
1. **Naive identity training은 실패** (PSNR vs noisy ≫ vs clean).
2. **Blind-spot masking trick**: 무작위 픽셀의 값을 주변에서 sampling한 값으로 교체 → 항등 학습 차단.
3. **N2V는 single noisy image만으로 신호를 회복** (PSNR이 Gaussian 보다 우월, oracle에 근접).
4. **잔차 (method noise) 가 거의 white noise** — 구조 보존 확인.

N2V의 결정적 가치: ground truth, paired data 둘 다 *없는* 데이터 (live-cell imaging, single-particle EM, 의료 영상 등)에 적용 가능.

This notebook empirically demonstrates N2V's four pillars: (1) naive identity training fails; (2) blind-spot masking blocks identity; (3) the trick recovers signal from a single noisy image; (4) residuals look like white noise (structure preserved). N2V is uniquely valuable for data with neither clean references nor paired noisy acquisitions — covering live-cell, single-particle EM, and medical imaging.